# **Earnings Momentum & Earnings Surprise (SUE)**

This study tests whether **post-earnings announcement drift (PEAD)** still shows up after **Halal screens**, and whether it is stronger, weaker, or similar versus **broad S&P 500 constituents**. After dropping banned businesses and names above the AAOIFI debt cap (`< 30%`), we keep stocks whose latest **standardized unanticipated earnings (SUE)** is **greater than 2** — actual EPS beats analyst consensus by more than two historical standard deviations of that stock's own forecast errors — and we **own them in proportion to company size** while the surprise is still inside a **60-trading-day PEAD window**.

**White-paper focus:** measure **PEAD efficacy in Halal equities vs. broad index constituents** — same SUE `> 2σ` rule, same holding window, two universes.

**What this notebook does, in plain terms:** start with US large caps, pull quarterly earnings surprises, standardize them, buy names that just crushed consensus, then compare a **Halal-screened book** with an **unscreened S&P 500 book**, plus event-study cumulative abnormal returns, against SPY, the S&P 500 index, and SPUS over **2020–2025**.


## Setup

Install dependencies and configure the backtest window.

`FAST_MODE = False` uses the full S&P 500 (slower: SEC / yfinance / earnings-calendar calls). Keep it `True` only for a smoke test. Performance stats start on the first day the strategy actually holds stocks, not on `START`.

SUE needs a history of forecast errors **before** each print: we download extra earnings dates and an extra price warmup so the 60-day drift window and rolling `σ` are defined.

**What the code cell does:**
- Installs `halalquant`, `yfinance`, and plotting libraries (including animation support).
- Sets the backtest dates (`START`, `END`) for **2020–2025** and strategy knobs: SUE threshold (`> 2σ`), PEAD holding window, rebalance frequency, and trading-cost assumptions.
- `FAST_MODE` shrinks the universe for a quick test; `MIN_HOLDINGS` is the minimum names needed before we invest that month.

**SEC fundamentals:** AAOIFI screens need SEC EDGAR history. The setup cell installs the local `halalquant` repo (`Development/halalquant`) when present, otherwise PyPI. Set `HALALQUANT_SEC_UA` to your name and email if SEC requests fail. **Restart the kernel** after the first setup run so imports pick up the updated package.


In [ ]:
import os
import subprocess
import sys
from datetime import date
from io import StringIO
from pathlib import Path

import requests
from IPython.display import HTML, display


def resolve_notebook_dir() -> Path:
    """Jupyter sometimes loses cwd(); find this notebook's folder reliably."""
    candidates = [Path.cwd()]
    candidates.append(
        Path.home() / "Documents/Development/Monterey-Finance/Research/papers/06-earn-momentum-sue"
    )
    for path in candidates:
        try:
            resolved = path.resolve()
        except OSError:
            continue
        if resolved.is_dir() and (resolved / "code.ipynb").exists():
            return resolved
    return Path.cwd()


NB_DIR = resolve_notebook_dir()
os.chdir(NB_DIR)

HQ_ROOT = NB_DIR.parents[3] / "halalquant"
PIP_DEPS = ["yfinance", "matplotlib", "requests", "lxml", "html5lib"]

if HQ_ROOT.is_dir() and (HQ_ROOT / "pyproject.toml").exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(HQ_ROOT)],
        cwd=str(NB_DIR),
        check=False,
    )
    hq_src = str(HQ_ROOT)
    if hq_src not in sys.path:
        sys.path.insert(0, hq_src)
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "halalquant"] + PIP_DEPS,
        cwd=str(NB_DIR),
        check=False,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + PIP_DEPS,
    cwd=str(NB_DIR),
    check=False,
)

os.environ.setdefault(
    "HALALQUANT_SEC_UA",
    "Monterey Finance Research halalquant/0.1.0 research@montereyfinance.com",
)

import warnings

import halalquant as hq
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from halalquant.providers._yfinance import _map_yahoo_activity, _ticker_info
from halalquant.screening._aaoifi import AAOIFIScreener
from halalquant.screening._sector_filter import SectorFilter
from matplotlib import animation
from matplotlib.ticker import PercentFormatter

warnings.filterwarnings("ignore", category=FutureWarning)

# --- Backtest configuration (2020–2025) ---
START = "2020-01-01"
END = "2025-12-31"
WARMUP_MONTHS = 18           # extra price history around events / PEAD window
EARNINGS_LIMIT = 40          # quarterly prints per name (~10 years)
SUE_THRESHOLD = 2.0          # actual EPS exceeds consensus by > 2σ
MIN_PRIOR_ERRORS = 6         # forecast-error history needed to estimate σ
HOLD_TRADING_DAYS = 60       # classic PEAD holding window after the print
REBALANCE_FREQ = "ME"
DEBT_THRESHOLD = 0.30        # AAOIFI debt / market-cap cap
MARKET_TICKER = "SPY"        # abnormal-return reference
FAST_MODE = True            # True → smaller sleeve (smoke test); False → full S&P 500
MIN_HOLDINGS = 3 if FAST_MODE else 5
RISK_FREE_RATE = 0.02
COST_BPS = 10
STRATEGY_LABEL = "Halal SUE > 2σ PEAD"
BROAD_LABEL = "Broad SUE > 2σ PEAD"  # same rule, no Halal financial screens

BENCHMARKS = {
    "SPY": "SPY",
    "S&P 500": "^GSPC",
    "SPUS": "SPUS",
}

print(f"notebook dir: {NB_DIR}")
print(f"halalquant v{hq.__version__} from {Path(hq.__file__).resolve().parent.parent}")
print(
    f"Window: {START} → {END}  FAST_MODE={FAST_MODE}  "
    f"SUE>{SUE_THRESHOLD}  HOLD={HOLD_TRADING_DAYS}d  MIN_HOLDINGS={MIN_HOLDINGS}"
)


## Step 1 — Universe: S&P 500 under Halal sector screens

We start from **S&P 500 constituents**, then apply `halalquant`'s sector screen (banks, alcohol, gambling, etc.). The **broad PEAD book** later uses the raw index list (minus names with no usable earnings). The **Halal PEAD book** uses only sector-eligible names that also pass AAOIFI financial ratios at each rebalance.

**What the code cell does:**
- `load_sp500_tickers()` — downloads the current S&P 500 symbol list (Wikipedia, with a small offline fallback).
- `apply_sector_screen()` — maps Yahoo sector/industry to Halal activity labels and drops banned businesses.
- Prints how many names survive into the Halal-eligible sleeve versus the broad index list.


In [ ]:
FALLBACK_SP500 = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "BRK-B", "LLY", "AVGO", "JPM",
    "UNH", "XOM", "V", "MA", "PG", "COST", "HD", "JNJ", "ABBV", "NFLX",
    "CRM", "MRK", "AMD", "PEP", "KO", "TMO", "ADBE", "WMT", "CSCO", "ACN",
    "MCD", "LIN", "ABT", "DHR", "INTC", "CMCSA", "TXN", "QCOM", "INTU", "AMAT",
    "DIS", "IBM", "GE", "CAT", "NOW", "VZ", "AMGN", "PFE", "ISRG", "GS",
]


def load_sp500_tickers() -> list[str]:
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "MontereyFinanceResearch/1.0 (halal-quant-notebook)"}
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        table = pd.read_html(StringIO(response.text), attrs={"id": "constituents"})[0]
        return table["Symbol"].astype(str).str.replace(".", "-", regex=False).tolist()
    except Exception as exc:
        print(f"Could not fetch S&P 500 list ({exc}); using fallback basket.")
        return FALLBACK_SP500.copy()


def apply_sector_screen(tickers: list[str]) -> tuple[list[str], pd.DataFrame, dict[str, str], pd.DataFrame]:
    sector_map: dict[str, str] = {}
    gics_rows: list[dict] = []
    n = len(tickers)
    for i, symbol in enumerate(tickers, 1):
        if i == 1 or i % 50 == 0 or i == n:
            print(f"  Sector / GICS labels {i}/{n}", flush=True)
        info = _ticker_info(yf.Ticker(symbol))
        y_sector = str(info.get("sector") or "")
        y_industry = str(info.get("industry") or "")
        mapped = _map_yahoo_activity(y_sector, y_industry)
        if mapped:
            sector_map[symbol] = mapped
        gics_rows.append(
            {
                "symbol": symbol,
                "gics_sector": y_sector or "Unclassified",
                "gics_industry": y_industry or "Unclassified",
            }
        )

    sector_filter = SectorFilter()
    kept = sector_filter.filter_symbols(tickers, sector_map=sector_map)
    audit = pd.DataFrame(sector_filter.audit_log)
    gics = pd.DataFrame(gics_rows).set_index("symbol")
    return kept, audit, sector_map, gics


raw_universe = load_sp500_tickers()
if FAST_MODE:
    raw_universe = raw_universe[:60]
    print("FAST_MODE is on — smoke-test universe only. Set FAST_MODE = False for the paper.\n")

print("Labeling sectors / industries …")
eligible_universe, sector_audit, activity_labels, gics = apply_sector_screen(raw_universe)
broad_universe = list(raw_universe)

print(f"Raw / broad universe:      {len(broad_universe)} tickers")
print(f"After Halal sector screen: {len(eligible_universe)} tickers")
print(f"Removed by sector:         {len(broad_universe) - len(eligible_universe)}")
print("\nSector mix after Halal activity screen:")
print(
    gics.loc[[s for s in eligible_universe if s in gics.index], "gics_sector"]
    .value_counts()
    .head(12)
    .to_string()
)

sector_audit.head(10)


## Steps 2–6 — Filter, score SUE, select, rebalance

Pipeline at each month-end rebalance (point-in-time, no look-ahead):

1. **FILTER** — Halal book only: AAOIFI financial screens (`debt < 30%`, cash & receivables caps) on the sector-eligible sleeve. Broad book skips this step.
2. **SCORE** — **SUE** for the latest print known on the rebalance date:
   \(\mathrm{SUE}_{i,t} = (EPS_{i,t} - \widehat{EPS}_{i,t}) / \sigma_{i,t}\)
   where \(\widehat{EPS}\) is the analyst consensus and \(\sigma_{i,t}\) is the standard deviation of that stock's **prior** (actual − consensus) errors.
3. **SELECT** — keep names with **SUE `> 2`** whose announcement is still inside the **60-trading-day PEAD window**; require at least `MIN_HOLDINGS` names.
4. **WEIGHT** — capitalisation weights within the selected basket (long-only).
5. **HOLD** — hold until the next month-end rebalance (names drop when the print ages out or SUE no longer qualifies).

**What the code cell does:**
- Helpers for prices, cap weights, SUE construction, portfolio construction, and performance stats.
- `select_sue_pead()` — optional Halal filter → live SUE `> 2σ` names still in the drift window.
- `run_sue_backtest()` — month-end rebalances and daily P&L.
- `event_car()` — cumulative abnormal returns vs SPY after each qualifying print (the PEAD event study).


In [ ]:
def price_on_or_before(price_panel: pd.DataFrame, symbol: str, as_of) -> float:
    if symbol not in price_panel.columns:
        return np.nan
    s = price_panel[symbol].dropna()
    s = s.loc[:pd.Timestamp(as_of)]
    return float(s.iloc[-1]) if not s.empty else np.nan


def _to_naive_ts(value) -> pd.Timestamp:
    ts = pd.Timestamp(value)
    if getattr(ts, "tzinfo", None) is not None:
        ts = ts.tz_convert("UTC").tz_localize(None)
    return ts.normalize()


def _col(frame: pd.DataFrame, *names: str) -> pd.Series | None:
    lookup = {str(c).strip().lower(): c for c in frame.columns}
    for name in names:
        hit = lookup.get(name.strip().lower())
        if hit is not None:
            return frame[hit]
    return None


def fetch_earnings_history(symbols: list[str], limit: int = EARNINGS_LIMIT) -> pd.DataFrame:
    """Quarterly consensus vs actual EPS from Yahoo earnings dates."""
    rows: list[dict] = []
    n = len(symbols)
    for i, symbol in enumerate(symbols, 1):
        if i == 1 or i % 25 == 0 or i == n:
            print(f"  Earnings calendar {i}/{n}", flush=True)
        try:
            ticker = yf.Ticker(symbol)
            try:
                raw = ticker.get_earnings_dates(limit=limit)
            except TypeError:
                raw = ticker.get_earnings_dates()
            if raw is None or not isinstance(raw, pd.DataFrame) or raw.empty:
                raw = getattr(ticker, "earnings_dates", None)
            if raw is None or not isinstance(raw, pd.DataFrame) or raw.empty:
                continue
        except Exception:
            continue

        frame = raw.copy()
        if not isinstance(frame.index, pd.RangeIndex):
            frame = frame.reset_index()
        date_col = None
        for candidate in frame.columns:
            key = str(candidate).strip().lower().replace("_", " ")
            if key in {"earnings date", "date", "index", "announce date"}:
                date_col = candidate
                break
        if date_col is None:
            # first datetime-like column
            for candidate in frame.columns:
                if pd.api.types.is_datetime64_any_dtype(frame[candidate]):
                    date_col = candidate
                    break
        if date_col is None:
            continue

        actual = _col(frame, "Reported EPS", "reported EPS", "epsActual", "reported_eps")
        estimate = _col(frame, "EPS Estimate", "epsEstimate", "eps_estimate")
        surprise = _col(frame, "Surprise(%)", "surprisePercent", "surprise_pct")
        if actual is None or estimate is None:
            continue

        for loc in frame.index:
            try:
                announce = _to_naive_ts(frame.at[loc, date_col])
            except Exception:
                continue
            act = pd.to_numeric(actual.at[loc], errors="coerce")
            est = pd.to_numeric(estimate.at[loc], errors="coerce")
            if pd.isna(act) or pd.isna(est):
                continue
            spr = np.nan
            if surprise is not None:
                spr = pd.to_numeric(surprise.at[loc], errors="coerce")
            rows.append(
                {
                    "symbol": symbol,
                    "announce_date": announce.date(),
                    "eps_actual": float(act),
                    "eps_estimate": float(est),
                    "ue": float(act) - float(est),
                    "surprise_pct": float(spr) if pd.notna(spr) else np.nan,
                }
            )

    if not rows:
        return pd.DataFrame(
            columns=["symbol", "announce_date", "eps_actual", "eps_estimate", "ue", "surprise_pct"]
        )
    out = pd.DataFrame(rows).drop_duplicates(subset=["symbol", "announce_date"])
    return out.sort_values(["symbol", "announce_date"]).reset_index(drop=True)


def attach_sue(earnings: pd.DataFrame, min_prior: int = MIN_PRIOR_ERRORS) -> pd.DataFrame:
    """Point-in-time SUE: (actual − consensus) / σ of *prior* forecast errors."""
    if earnings.empty:
        out = earnings.copy()
        out["sue"] = pd.Series(dtype=float)
        out["sigma_ue"] = pd.Series(dtype=float)
        out["n_prior"] = pd.Series(dtype=int)
        return out

    pieces: list[pd.DataFrame] = []
    for symbol, grp in earnings.groupby("symbol", sort=False):
        g = grp.sort_values("announce_date").copy()
        ue = g["ue"].astype(float)
        sigmas, sues, priors = [], [], []
        for i in range(len(g)):
            hist = ue.iloc[:i].dropna()
            n_prior = int(len(hist))
            priors.append(n_prior)
            if n_prior < min_prior:
                sigmas.append(np.nan)
                sues.append(np.nan)
                continue
            sigma = float(hist.std(ddof=1))
            if not np.isfinite(sigma) or sigma <= 1e-8:
                sigmas.append(np.nan)
                sues.append(np.nan)
                continue
            sigmas.append(sigma)
            sues.append(float(ue.iloc[i]) / sigma)
        g["sigma_ue"] = sigmas
        g["sue"] = sues
        g["n_prior"] = priors
        pieces.append(g)
    return pd.concat(pieces, ignore_index=True)


def trading_days_between(price_index: pd.DatetimeIndex, start, end) -> int:
    """Inclusive count of sessions with start < d <= end (post-announcement window)."""
    lo = pd.Timestamp(start)
    hi = pd.Timestamp(end)
    if hi <= lo:
        return 0
    window = price_index[(price_index > lo) & (price_index <= hi)]
    return int(len(window))


def assign_cap_weights(selected: pd.DataFrame) -> pd.DataFrame:
    out = selected.copy()
    if out.empty:
        out["weight"] = pd.Series(dtype=float)
        return out
    cap = pd.to_numeric(out["market_cap"], errors="coerce").clip(lower=0)
    total = float(cap.sum(skipna=True))
    if np.isfinite(total) and total > 0:
        out["weight"] = cap / total
        out["weight"] = out["weight"].fillna(0.0)
        wsum = float(out["weight"].sum())
        out["weight"] = out["weight"] / wsum if wsum > 0 else 1.0 / len(out)
    else:
        out["weight"] = 1.0 / len(out)
    return out


def latest_events_as_of(sue_events: pd.DataFrame, as_of: date) -> pd.DataFrame:
    """Most recent print per symbol known on as_of (announcement already public)."""
    if sue_events.empty:
        return sue_events.iloc[0:0].copy()
    snap = sue_events[pd.to_datetime(sue_events["announce_date"]).dt.date <= as_of].copy()
    if snap.empty:
        return snap
    snap = snap.sort_values(["symbol", "announce_date"])
    return snap.groupby("symbol", as_index=False).tail(1)


def select_sue_pead(
    panel: pd.DataFrame,
    sue_events: pd.DataFrame,
    price_index: pd.DatetimeIndex,
    as_of: date,
    apply_halal: bool = True,
    sue_threshold: float = SUE_THRESHOLD,
    hold_days: int = HOLD_TRADING_DAYS,
    min_holdings: int = MIN_HOLDINGS,
    screener: AAOIFIScreener | None = None,
) -> pd.DataFrame:
    empty_cols = [
        "symbol", "sue", "ue", "sigma_ue", "announce_date", "days_since_print",
        "market_cap", "weight", "debt_ratio", "cash_ratio", "receivables_ratio",
        "eps_actual", "eps_estimate",
    ]
    as_of_d = pd.Timestamp(as_of).date()
    live = latest_events_as_of(sue_events, as_of_d)
    if live.empty:
        return pd.DataFrame(columns=empty_cols)

    live["days_since_print"] = live["announce_date"].map(
        lambda d: trading_days_between(price_index, d, as_of_d)
    )
    live = live[
        live["sue"].notna()
        & np.isfinite(live["sue"])
        & (live["sue"] > sue_threshold)
        & (live["days_since_print"] >= 1)
        & (live["days_since_print"] <= hold_days)
    ].copy()
    if live.empty:
        return pd.DataFrame(columns=empty_cols)

    if apply_halal:
        screener = screener or AAOIFIScreener(debt_threshold=DEBT_THRESHOLD)
        snap = panel[pd.to_datetime(panel["as_of"]).dt.date == as_of_d].copy()
        if snap.empty:
            return pd.DataFrame(columns=empty_cols)
        screened = screener.evaluate_compliance(snap)
        snap = snap.drop(
            columns=[c for c in ("debt_ratio", "cash_ratio", "receivables_ratio") if c in snap.columns]
        )
        passed = snap.merge(screened, on="symbol", how="inner")
        passed = passed[passed["is_compliant"].fillna(False)].copy()
        passed = passed[pd.to_numeric(passed["market_cap"], errors="coerce") > 0]
        live = live.merge(
            passed[["symbol", "market_cap", "debt_ratio", "cash_ratio", "receivables_ratio"]],
            on="symbol",
            how="inner",
        )
    else:
        snap = panel[pd.to_datetime(panel["as_of"]).dt.date == as_of_d].copy()
        caps = snap[["symbol", "market_cap"]].drop_duplicates("symbol") if not snap.empty else pd.DataFrame()
        if caps.empty:
            live["market_cap"] = np.nan
        else:
            live = live.merge(caps, on="symbol", how="left")
        live["debt_ratio"] = np.nan
        live["cash_ratio"] = np.nan
        live["receivables_ratio"] = np.nan
        live = live[pd.to_numeric(live["market_cap"], errors="coerce") > 0]

    if live.empty:
        return pd.DataFrame(columns=empty_cols)

    selected = assign_cap_weights(live)
    keep = [c for c in empty_cols if c in selected.columns]
    return selected[keep].sort_values("weight", ascending=False)


def run_sue_backtest(
    metrics: pd.DataFrame,
    sue_events: pd.DataFrame,
    prices: pd.DataFrame,
    apply_halal: bool = True,
    min_holdings: int = MIN_HOLDINGS,
) -> tuple[pd.DataFrame, pd.DataFrame, list[date]]:
    screener = AAOIFIScreener(debt_threshold=DEBT_THRESHOLD) if apply_halal else None

    price_panel = (
        prices.pivot(index="date", columns="symbol", values="adj_close")
        .sort_index()
        .ffill()
    )
    price_panel.index = pd.to_datetime(price_panel.index)
    daily_returns = price_panel.pct_change(fill_method=None)

    rebalance_dates = sorted(pd.to_datetime(metrics["as_of"].dropna().unique()).date)
    weights_by_date: dict[date, pd.Series] = {}
    selection_log: list[pd.DataFrame] = []

    for as_of in rebalance_dates:
        picks = select_sue_pead(
            metrics,
            sue_events,
            price_panel.index,
            as_of=as_of,
            apply_halal=apply_halal,
            min_holdings=min_holdings,
            screener=screener,
        )
        if picks.empty:
            weights_by_date[as_of] = pd.Series(dtype=float)
        else:
            weights_by_date[as_of] = picks.set_index("symbol")["weight"].astype(float)
            selection_log.append(picks.assign(as_of=as_of))

    selection_df = (
        pd.concat(selection_log, ignore_index=True)
        if selection_log
        else pd.DataFrame(columns=["symbol", "sue", "weight", "as_of"])
    )

    port_rows: list[dict] = []
    active_w = pd.Series(dtype=float)
    last_rebalance: date | None = None
    live = False

    for dt in daily_returns.index:
        if dt.date() < pd.Timestamp(START).date():
            continue

        prior = [d for d in rebalance_dates if pd.Timestamp(d) <= dt]
        if prior and prior[-1] != last_rebalance:
            active_w = weights_by_date.get(prior[-1], pd.Series(dtype=float))
            last_rebalance = prior[-1]
            if not active_w.empty:
                live = True

        if not live:
            port_rows.append({"date": dt.date(), "return": np.nan, "n_holdings": 0})
            continue

        if active_w.empty:
            port_rows.append({"date": dt.date(), "return": 0.0, "n_holdings": 0})
            continue

        held = [s for s in active_w.index if s in daily_returns.columns]
        w = active_w.reindex(held).astype(float)
        w = w[w > 0]
        if w.empty:
            port_rows.append({"date": dt.date(), "return": 0.0, "n_holdings": 0})
            continue
        w = w / w.sum()
        day_ret = (daily_returns.loc[dt, w.index] * w).sum(skipna=True)
        port_rows.append({
            "date": dt.date(),
            "return": 0.0 if pd.isna(day_ret) else float(day_ret),
            "n_holdings": int(len(w)),
        })

    port_returns = pd.DataFrame(port_rows)
    port_returns["date"] = pd.to_datetime(port_returns["date"])
    return port_returns, selection_df, rebalance_dates


def calc_performance_stats(daily_returns: pd.Series, label: str, rf: float = RISK_FREE_RATE) -> dict:
    r = daily_returns.dropna()
    if r.empty:
        return {
            "Strategy": label, "CAGR": np.nan, "Volatility": np.nan, "Sharpe": np.nan,
            "Sortino": np.nan, "Max Drawdown": np.nan, "Calmar": np.nan,
        }
    equity = (1 + r).cumprod()
    years = len(r) / 252
    cagr = equity.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan
    vol = r.std() * np.sqrt(252)
    excess = r - rf / 252
    sharpe = excess.mean() / excess.std() * np.sqrt(252) if excess.std() > 0 else np.nan
    downside = r[r < 0]
    sortino = (r.mean() - rf / 252) / downside.std() * np.sqrt(252) if len(downside) else np.nan
    dd = equity / equity.cummax() - 1
    max_dd = dd.min()
    calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan
    return {
        "Strategy": label,
        "CAGR": cagr,
        "Volatility": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Max Drawdown": max_dd,
        "Calmar": calmar,
    }


def event_car(
    events: pd.DataFrame,
    price_panel: pd.DataFrame,
    market: str = MARKET_TICKER,
    horizons: tuple[int, ...] = (1, 5, 21, 60),
) -> pd.DataFrame:
    """Mean cumulative abnormal return vs SPY after SUE > 2σ prints."""
    if market not in price_panel.columns:
        return pd.DataFrame()
    mkt = price_panel[market].pct_change(fill_method=None)
    idx = price_panel.index
    rows = []
    for ev in events.itertuples(index=False):
        sym = ev.symbol
        if sym not in price_panel.columns:
            continue
        announce = pd.Timestamp(ev.announce_date)
        after = idx[idx > announce]
        if after.empty:
            continue
        stock = price_panel[sym].pct_change(fill_method=None)
        abn = (stock - mkt).reindex(after)
        rec = {
            "symbol": sym,
            "announce_date": pd.Timestamp(ev.announce_date).date(),
            "sue": float(ev.sue) if pd.notna(ev.sue) else np.nan,
        }
        csum = abn.cumsum()
        for h in horizons:
            rec[f"CAR_{h}"] = float(csum.iloc[h - 1]) if len(csum) >= h else np.nan
        rec["n_days"] = int(abn.dropna().shape[0])
        rows.append(rec)
    return pd.DataFrame(rows)


print("Helpers ready: attach_sue, select_sue_pead, run_sue_backtest, event_car")


### Download prices, fundamentals, and earnings surprises

**What this code cell does:**
- Pulls point-in-time AAOIFI financial snapshots (needed for the Halal book) at each month-end.
- Downloads Yahoo earnings dates (actual vs consensus EPS) for the **broad** index list, then computes point-in-time SUE.
- Downloads adjusted prices for all names plus SPY / `^GSPC` / SPUS, starting earlier than `START` so the PEAD window has enough history.


In [ ]:
price_start = (pd.Timestamp(START) - pd.DateOffset(months=WARMUP_MONTHS)).strftime("%Y-%m-%d")
metrics_universe = sorted(set(eligible_universe) | set(broad_universe))

print("Fetching point-in-time AAOIFI metrics …")
metrics = hq.get_financial_metrics(
    metrics_universe,
    start=START,
    end=END,
    freq=REBALANCE_FREQ,
)
print(f"  {metrics.shape[0]} snapshot rows")

print("Fetching earnings calendars (actual vs consensus) …")
earnings_raw = fetch_earnings_history(broad_universe, limit=EARNINGS_LIMIT)
sue_events = attach_sue(earnings_raw, min_prior=MIN_PRIOR_ERRORS)
print(f"  {len(earnings_raw)} prints with both actual and estimate")
print(f"  {sue_events['sue'].notna().sum()} prints with a usable SUE (prior n ≥ {MIN_PRIOR_ERRORS})")
if not sue_events.empty:
    live_prints = sue_events[
        (pd.to_datetime(sue_events["announce_date"]) >= pd.Timestamp(START))
        & (pd.to_datetime(sue_events["announce_date"]) <= pd.Timestamp(END))
    ]
    n_big = int((live_prints["sue"] > SUE_THRESHOLD).sum())
    print(
        f"  Live window prints: {len(live_prints)}  |  SUE > {SUE_THRESHOLD:g}: {n_big}  "
        f"({n_big / max(len(live_prints), 1):.1%} of dated prints with SUE)"
    )

# halalquant.validate_symbols rejects index tickers like ^GSPC — fetch those via yfinance.
hq_ok = {t for t in set(BENCHMARKS.values()) | {MARKET_TICKER} if t.replace(".", "").replace("-", "").isalnum()}
index_tickers = sorted(set(BENCHMARKS.values()) - hq_ok)
all_symbols = sorted(set(broad_universe) | hq_ok)

print(f"Downloading price history from {price_start} …")
prices = hq.download(all_symbols, start=price_start, end=END)
prices["date"] = pd.to_datetime(prices["date"])

if index_tickers:
    print(f"  Fetching index benchmarks via yfinance: {index_tickers}")
    raw = yf.download(
        index_tickers,
        start=price_start,
        end=END,
        auto_adjust=False,
        progress=False,
        group_by="ticker",
        threads=False,
    )
    index_frames = []
    for ticker in index_tickers:
        if isinstance(raw.columns, pd.MultiIndex):
            if ticker not in raw.columns.get_level_values(0):
                print(f"  Warning: no data for {ticker}")
                continue
            sub = raw[ticker].copy()
        else:
            sub = raw.copy()
        sub = sub.rename(columns=str.title)
        adj = sub["Adj Close"] if "Adj Close" in sub.columns else sub["Close"]
        dates = pd.to_datetime(sub.index)
        if getattr(dates, "tz", None) is not None:
            dates = dates.tz_convert("UTC").tz_localize(None)
        frame = pd.DataFrame({
            "symbol": ticker,
            "date": dates,
            "open": sub.get("Open"),
            "high": sub.get("High"),
            "low": sub.get("Low"),
            "close": sub.get("Close"),
            "volume": sub.get("Volume"),
            "adj_close": adj,
        })
        index_frames.append(frame.dropna(subset=["adj_close"]))
    if index_frames:
        prices = pd.concat([prices, *index_frames], ignore_index=True)

print(f"  {prices['symbol'].nunique()} symbols, {prices['date'].nunique()} trading days")

display(sue_events.dropna(subset=["sue"]).head())


### Run the backtest

**What this code cell does:**
- Sanity-checks the last rebalance funnel: fundamentals → Halal → SUE `> 2σ` PEAD picks.
- Runs the main **Halal SUE PEAD** strategy and the **broad-index SUE PEAD** robustness book (same surprise rule, no AAOIFI / sector financial overlay at selection).
- Prints holdings summary and the latest Halal portfolio weights.


In [ ]:
_price_panel = (
    prices.pivot(index="date", columns="symbol", values="adj_close")
    .sort_index()
    .ffill()
)
_price_panel.index = pd.to_datetime(_price_panel.index)
_as_of = sorted(pd.to_datetime(metrics["as_of"].dropna().unique()).date)[-1]
_screener = AAOIFIScreener(debt_threshold=DEBT_THRESHOLD)
_snap = metrics[pd.to_datetime(metrics["as_of"]).dt.date == _as_of]
_halal = _snap.merge(_screener.evaluate_compliance(_snap), on="symbol", how="inner")
_halal = _halal[_halal["is_compliant"].fillna(False)]
_scored = select_sue_pead(metrics, sue_events, _price_panel.index, _as_of, apply_halal=True)
_broad_picks = select_sue_pead(metrics, sue_events, _price_panel.index, _as_of, apply_halal=False)
print(
    f"Funnel on {_as_of}: fundamentals={len(_snap)}  Halal={len(_halal)}  "
    f"Halal SUE>2 picks={len(_scored)}  Broad SUE>2 picks={len(_broad_picks)}  "
    f"MIN_HOLDINGS={MIN_HOLDINGS}"
)
if not _scored.empty:
    print(
        f"Halal SUE range of picks: {_scored['sue'].min():.2f} → {_scored['sue'].max():.2f}  "
        f"(median {_scored['sue'].median():.2f})"
    )

strategy_returns, selections, rebalance_dates = run_sue_backtest(
    metrics=metrics,
    sue_events=sue_events[sue_events["symbol"].isin(eligible_universe)],
    prices=prices[prices["symbol"].isin(set(eligible_universe) | {MARKET_TICKER})],
    apply_halal=True,
)

broad_returns, broad_selections, _ = run_sue_backtest(
    metrics=metrics,
    sue_events=sue_events,
    prices=prices[prices["symbol"].isin(set(broad_universe) | {MARKET_TICKER})],
    apply_halal=False,
)

live_dates = strategy_returns.loc[strategy_returns["return"].notna(), "date"]
holdings_per_date = selections.groupby("as_of")["symbol"].nunique() if not selections.empty else pd.Series(dtype=int)

print(f"Rebalance dates with data: {len(rebalance_dates)}")
print(f"First invested date:       {live_dates.min().date() if not live_dates.empty else 'n/a'}")
print(f"Last rebalance:            {rebalance_dates[-1] if rebalance_dates else 'n/a'}")
if not holdings_per_date.empty:
    print(
        "Halal holdings per live rebalance: "
        f"min={int(holdings_per_date.min())}  "
        f"median={holdings_per_date.median():.0f}  "
        f"max={int(holdings_per_date.max())}"
    )
if not broad_selections.empty:
    bh = broad_selections.groupby("as_of")["symbol"].nunique()
    print(
        "Broad holdings per live rebalance: "
        f"min={int(bh.min())}  median={bh.median():.0f}  max={int(bh.max())}"
    )

print(f"\nLatest Halal holdings ({rebalance_dates[-1] if rebalance_dates else 'n/a'}) — largest weights first:")
if rebalance_dates and not selections.empty:
    latest = selections[selections["as_of"] == rebalance_dates[-1]].copy()
    show = latest.copy()
    for col in ("sue", "weight", "debt_ratio", "days_since_print"):
        if col not in show.columns:
            continue
        if col == "sue":
            show[col] = show[col].map(lambda x: f"{x:.2f}" if pd.notna(x) else "—")
        elif col == "days_since_print":
            show[col] = show[col].map(lambda x: f"{int(x)}" if pd.notna(x) else "—")
        else:
            show[col] = show[col].map(lambda x: f"{x:.2%}" if pd.notna(x) else "—")
    display(show.reset_index(drop=True))
else:
    print("  (no compliant SUE > 2σ picks on last rebalance date)")


## Step 7 — Performance vs SPY, S&P 500, and SPUS

**What the code cell does:**
- Aligns Halal and broad PEAD daily returns with SPY, `^GSPC` (S&P 500), and SPUS over the live window.
- Reports CAGR, volatility, Sharpe, Sortino, max drawdown, and Calmar.
- Plots equity curves and rolling 12-month excess return vs SPY.


In [ ]:
def benchmark_returns(ticker: str) -> pd.Series:
    px = prices[prices["symbol"] == ticker].copy()
    px["date"] = pd.to_datetime(px["date"])
    px = px.sort_values("date").set_index("date")["adj_close"]
    s = px.pct_change(fill_method=None)
    return s.loc[s.index >= pd.Timestamp(START)]


def align_to_live(strategy: pd.Series, benches: dict[str, pd.Series]) -> pd.DataFrame:
    strategy = strategy.copy()
    strategy.index = pd.to_datetime(strategy.index)
    panel = pd.DataFrame({"strategy": strategy})
    for name, series in benches.items():
        s = series.copy()
        s.index = pd.to_datetime(s.index)
        panel[name] = s
    return panel.dropna(how="any")


fig_dir = Path.cwd() / "figures"
if not (Path.cwd() / "code.ipynb").exists():
    fig_dir = Path.cwd() / "Research" / "papers" / "06-earn-momentum-sue" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

bench_series = {name: benchmark_returns(ticker) for name, ticker in BENCHMARKS.items()}
strategy_s = strategy_returns.set_index("date")["return"].rename(STRATEGY_LABEL)
strategy_s.index = pd.to_datetime(strategy_s.index)
broad_s = broad_returns.set_index("date")["return"].rename(BROAD_LABEL)
broad_s.index = pd.to_datetime(broad_s.index)

main_panel = align_to_live(strategy_s, bench_series).rename(columns={"strategy": STRATEGY_LABEL})
broad_panel = align_to_live(broad_s, bench_series).rename(columns={"strategy": BROAD_LABEL})

if main_panel.empty:
    raise RuntimeError("No overlapping live days between the Halal strategy and benchmarks.")

print(
    "Live window (Halal): "
    f"{main_panel.index.min().date()} → {main_panel.index.max().date()}  "
    f"({len(main_panel)} trading days)"
)

rows = [calc_performance_stats(main_panel[STRATEGY_LABEL], STRATEGY_LABEL)]
if not broad_panel.empty:
    rows.append(calc_performance_stats(broad_panel[BROAD_LABEL], BROAD_LABEL))
for name in BENCHMARKS:
    rows.append(calc_performance_stats(main_panel[name], name))

performance = pd.DataFrame(rows).set_index("Strategy")
formatters = {
    "CAGR": "{:.2%}",
    "Volatility": "{:.2%}",
    "Sharpe": "{:.2f}",
    "Sortino": "{:.2f}",
    "Max Drawdown": "{:.2%}",
    "Calmar": "{:.2f}",
}
display(performance.style.format(formatters))

plot_panel = main_panel.copy()
if not broad_panel.empty:
    plot_panel = plot_panel.join(broad_panel[[BROAD_LABEL]], how="left")
equity_curves = (1 + plot_panel).cumprod()
equity_curves.plot(
    figsize=(11, 5),
    title="Halal vs broad SUE > 2σ PEAD vs SPY / S&P 500 / SPUS (2020–2025)",
)
plt.ylabel("Growth of $1")
plt.xlabel("")
plt.grid(alpha=0.3)
plt.legend(loc="upper left")
plt.tight_layout()
plt.savefig(fig_dir / "equity-curves.png", dpi=140)
plt.show()

if "SPY" in main_panel.columns:
    rolling_excess = (
        (1 + main_panel[STRATEGY_LABEL]).rolling(252).apply(np.prod, raw=True)
        - (1 + main_panel["SPY"]).rolling(252).apply(np.prod, raw=True)
    )
    rolling_excess.plot(figsize=(11, 3.5), color="tab:green", title="Rolling 12-Month Excess Return vs SPY (Halal PEAD)")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.ylabel("Excess return")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(fig_dir / "rolling-excess.png", dpi=140)
    plt.show()


## Step 8 — PEAD event study: Halal vs broad index

White-paper focus: after a **SUE `> 2σ`** print, does the stock still drift, and is that drift **different inside the Halal sleeve** than in the **full S&P 500**?

Abnormal return is daily stock return minus SPY. We report mean **CAR** at 1, 5, 21, and 60 trading days after the announcement (the announcement session itself is skipped so overnight gaps do not dominate).

**What the code cell does:**
- Builds the SUE `> 2σ` event list in the live window for Halal-eligible names vs all index names.
- Tags Halal compliance at the month-end snapshot on or before the print.
- Plots mean CAR paths, drawdowns of the two books, calendar-year returns, and average SUE / holdings through time.


In [ ]:
live_mask = (
    (pd.to_datetime(sue_events["announce_date"]) >= pd.Timestamp(START))
    & (pd.to_datetime(sue_events["announce_date"]) <= pd.Timestamp(END))
    & sue_events["sue"].notna()
    & (sue_events["sue"] > SUE_THRESHOLD)
)
pos_events = sue_events.loc[live_mask].copy()

halal_set = set(eligible_universe)
broad_events = pos_events[pos_events["symbol"].isin(broad_universe)].copy()
halal_sector_events = pos_events[pos_events["symbol"].isin(halal_set)].copy()

# Point-in-time AAOIFI flag: latest month-end snapshot on/before each print
comp_rows = []
if not metrics.empty:
    met = metrics.copy()
    met["as_of"] = pd.to_datetime(met["as_of"])
    screener = AAOIFIScreener(debt_threshold=DEBT_THRESHOLD)
    for as_of, snap in met.groupby(met["as_of"].dt.date):
        screened = screener.evaluate_compliance(snap)
        tmp = screened[["symbol", "is_compliant"]].copy()
        tmp["as_of"] = as_of
        comp_rows.append(tmp)
comp_hist = pd.concat(comp_rows, ignore_index=True) if comp_rows else pd.DataFrame(columns=["as_of", "symbol", "is_compliant"])


def is_halal_at_print(symbol: str, announce) -> bool:
    if symbol not in halal_set or comp_hist.empty:
        return False
    cutoff = pd.Timestamp(announce).date()
    prior = comp_hist[
        (comp_hist["symbol"] == symbol)
        & (pd.to_datetime(comp_hist["as_of"]).dt.date <= cutoff)
    ]
    if prior.empty:
        return False
    prior = prior.sort_values("as_of")
    return bool(prior.iloc[-1]["is_compliant"])


halal_fin_events = halal_sector_events[
    [
        is_halal_at_print(r.symbol, r.announce_date)
        for r in halal_sector_events.itertuples(index=False)
    ]
].copy() if not halal_sector_events.empty else halal_sector_events

print(
    f"SUE > {SUE_THRESHOLD:g} prints in {START[:4]}–{END[:4]}: "
    f"broad={len(broad_events)}  sector-eligible={len(halal_sector_events)}  "
    f"AAOIFI-compliant at print={len(halal_fin_events)}"
)

car_broad = event_car(broad_events, _price_panel)
car_halal = event_car(halal_fin_events if not halal_fin_events.empty else halal_sector_events, _price_panel)

horizons = [1, 5, 21, 60]


def car_summary(car_df: pd.DataFrame, label: str) -> dict:
    row = {"Universe": label, "N events": int(len(car_df))}
    for h in horizons:
        col = f"CAR_{h}"
        if car_df.empty or col not in car_df.columns:
            row[col] = np.nan
            continue
        row[col] = float(car_df[col].mean())
    return row


pead_table = pd.DataFrame([
    car_summary(car_halal, "Halal (AAOIFI at print)"),
    car_summary(car_broad, "Broad S&P 500"),
]).set_index("Universe")
display(pead_table.style.format({**{f"CAR_{h}": "{:.2%}" for h in horizons}, "N events": "{:.0f}"}))


def mean_car_path(events: pd.DataFrame, max_h: int = 60) -> pd.Series:
    if events.empty:
        return pd.Series(dtype=float)
    mkt = _price_panel[MARKET_TICKER].pct_change(fill_method=None)
    idx = _price_panel.index
    acc = []
    for ev in events.itertuples(index=False):
        if ev.symbol not in _price_panel.columns:
            continue
        after = idx[idx > pd.Timestamp(ev.announce_date)][:max_h]
        if after.empty:
            continue
        stock = _price_panel[ev.symbol].pct_change(fill_method=None)
        abn = (stock - mkt).reindex(after).astype(float)
        path = abn.cumsum().reset_index(drop=True)
        path.index = path.index + 1
        acc.append(path)
    if not acc:
        return pd.Series(dtype=float)
    return pd.concat(acc, axis=1).mean(axis=1)


path_h = mean_car_path(halal_fin_events if not halal_fin_events.empty else halal_sector_events)
path_b = mean_car_path(broad_events)
pead_paths = pd.DataFrame({"Halal": path_h, "Broad index": path_b}).dropna(how="all")
if not pead_paths.empty:
    pead_paths.plot(figsize=(11, 4.5), title="Mean CAR vs SPY after SUE > 2σ prints (PEAD)")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.xlabel("Trading days after announcement")
    plt.ylabel("Mean CAR")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(fig_dir / "pead-car.png", dpi=140)
    plt.show()

# Drawdowns + calendar years for the two books vs markets
equity_live = (1 + plot_panel).cumprod()
drawdowns = equity_live / equity_live.cummax() - 1
drawdowns.plot(figsize=(11, 4), title="Drawdowns — Halal PEAD vs broad PEAD vs markets")
plt.ylabel("Drawdown")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(fig_dir / "drawdowns.png", dpi=140)
plt.show()

cal = plot_panel.copy()
cal["year"] = cal.index.year
year_rets = (1 + cal).groupby("year").prod() - 1
if "year" in year_rets.columns:
    year_rets = year_rets.drop(columns=["year"], errors="ignore")
year_rets.plot(kind="bar", figsize=(11, 4), title="Calendar-year returns")
plt.ylabel("Return")
plt.axhline(0, color="black", linewidth=0.8)
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(fig_dir / "calendar-year-returns.png", dpi=140)
plt.show()
display(year_rets.style.format("{:.1%}"))

if not selections.empty:
    sue_ts = selections.groupby("as_of").apply(
        lambda d: np.average(d["sue"], weights=d["weight"]) if d["weight"].sum() > 0 else d["sue"].mean(),
        include_groups=False,
    )
    n_ts = selections.groupby("as_of")["symbol"].nunique()
    fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
    sue_ts.plot(ax=ax[0], color="tab:red", title="Portfolio weighted-average SUE (month-end)")
    ax[0].axhline(SUE_THRESHOLD, color="black", linewidth=0.8, linestyle="--", label=f"SUE = {SUE_THRESHOLD:g}")
    ax[0].set_ylabel("SUE")
    ax[0].legend(loc="upper right")
    ax[0].grid(alpha=0.3)
    n_ts.plot(ax=ax[1], color="tab:blue", title="Halal PEAD holdings count")
    ax[1].set_ylabel("Names")
    ax[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(fig_dir / "sue-holdings-path.png", dpi=140)
    plt.show()


## Strategy demonstration (animated) — 2020–2025

**One cell** that plays the live backtest through time: cumulative returns of the **Halal SUE PEAD** book versus **SPY**, **S&P 500 (`^GSPC`)**, and **SPUS**, alongside trailing realized volatility for each series.

Run this cell after the backtest and performance panels are in memory. The animation is self-contained (uses `main_panel` / `STRATEGY_LABEL`).


In [ ]:
# =============================================================================
# ANIMATED BACKTEST DEMO — returns + volatility vs SPY / S&P 500 / SPUS
# Requires: main_panel, STRATEGY_LABEL, BENCHMARKS, fig_dir from prior cells
# =============================================================================

from IPython.display import HTML, display

anim_cols = [STRATEGY_LABEL] + [c for c in ["SPY", "S&P 500", "SPUS"] if c in main_panel.columns]
anim_rets = main_panel[anim_cols].dropna(how="any").copy()
if anim_rets.empty:
    raise RuntimeError("main_panel has no overlapping returns for the animation.")

anim_equity = (1 + anim_rets).cumprod()
anim_vol = anim_rets.rolling(63).std() * np.sqrt(252)

step = max(1, len(anim_equity) // 180)
frame_idx = list(range(0, len(anim_equity), step))
if frame_idx[-1] != len(anim_equity) - 1:
    frame_idx.append(len(anim_equity) - 1)

colors = {
    STRATEGY_LABEL: "#c0392b",
    "SPY": "#2c3e50",
    "S&P 500": "#7f8c8d",
    "SPUS": "#2980b9",
}

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
fig.suptitle("Halal SUE > 2σ PEAD — animated path vs markets (2020–2025)", fontsize=13, fontweight="bold")

lines_eq, lines_vol = {}, {}
for col in anim_cols:
    (lines_eq[col],) = axes[0].plot([], [], lw=2.0, color=colors.get(col, None), label=col)
    (lines_vol[col],) = axes[1].plot([], [], lw=1.8, color=colors.get(col, None), label=col)

axes[0].set_ylabel("Growth of $1")
axes[0].set_xlim(anim_equity.index.min(), anim_equity.index.max())
ymin0, ymax0 = float(anim_equity.min().min()) * 0.95, float(anim_equity.max().max()) * 1.05
axes[0].set_ylim(ymin0, ymax0)
axes[0].grid(alpha=0.3)
axes[0].legend(loc="upper left", fontsize=9)
axes[0].set_title("Cumulative returns")

axes[1].set_ylabel("Trailing 63d vol (ann.)")
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[1].set_xlim(anim_vol.index.min(), anim_vol.index.max())
vol_plot = anim_vol.dropna(how="all")
ymin1 = float(vol_plot.min().min()) * 0.9 if not vol_plot.empty else 0
ymax1 = float(vol_plot.max().max()) * 1.1 if not vol_plot.empty else 0.5
axes[1].set_ylim(max(0, ymin1), ymax1)
axes[1].grid(alpha=0.3)
axes[1].legend(loc="upper left", fontsize=9)
axes[1].set_title("Realized volatility")

date_text = axes[0].text(
    0.02, 0.05, "", transform=axes[0].transAxes, fontsize=10,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
)
stats_text = axes[1].text(
    0.98, 0.95, "", transform=axes[1].transAxes, fontsize=9, ha="right", va="top",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
)


def _init_anim():
    for col in anim_cols:
        lines_eq[col].set_data([], [])
        lines_vol[col].set_data([], [])
    date_text.set_text("")
    stats_text.set_text("")
    return list(lines_eq.values()) + list(lines_vol.values()) + [date_text, stats_text]


def _update_anim(frame_i: int):
    i = frame_idx[frame_i]
    x = anim_equity.index[: i + 1]
    for col in anim_cols:
        lines_eq[col].set_data(x, anim_equity[col].iloc[: i + 1].values)
        yv = anim_vol[col].iloc[: i + 1]
        lines_vol[col].set_data(anim_vol.index[: i + 1], yv.values)
    asof = anim_equity.index[i]
    date_text.set_text(asof.strftime("%Y-%m-%d"))
    bits = [f"As of {asof.date()}"]
    for col in anim_cols:
        cum = float(anim_equity[col].iloc[i] - 1)
        vol = anim_vol[col].iloc[i]
        vol_s = f"{vol:.0%}" if pd.notna(vol) else "—"
        bits.append(f"{col}: {cum:+.1%} | σ {vol_s}")
    stats_text.set_text("\n".join(bits))
    return list(lines_eq.values()) + list(lines_vol.values()) + [date_text, stats_text]


anim = animation.FuncAnimation(
    fig,
    _update_anim,
    init_func=_init_anim,
    frames=len(frame_idx),
    interval=80,
    blit=False,
    repeat=True,
)
plt.tight_layout()

fig.savefig(fig_dir / "animated-demo-final.png", dpi=140)
html_anim = anim.to_jshtml()
display(HTML(html_anim))
plt.close(fig)

end_stats = pd.DataFrame({
    "Total Return": anim_equity.iloc[-1] - 1,
    "Ann. Vol (full)": anim_rets.std() * np.sqrt(252),
    "Final 63d Vol": anim_vol.iloc[-1],
})
print("\nEnd-of-window comparison (animation universe):")
display(end_stats.style.format("{:.2%}"))


## Step 9 — Does larger SUE actually produce more drift?

We sort every usable print (Halal-eligible names) into **SUE quintiles** and track cap-weighted 60-day CAR vs SPY. Monotonic Q5 > Q1 supports an earnings-surprise premium; a flat pattern would suggest the `> 2σ` cutoff is arbitrary, or that PEAD is just a sector/size effect.

A second plot shows **average GICS sector weights** in the live Halal PEAD book — the usual check that the edge is not only a mega-cap tech overweight.


In [ ]:
def cap_weighted_returns(weights_by_date: dict[date, pd.Series], daily_returns: pd.DataFrame) -> pd.Series:
    rebal = sorted(weights_by_date.keys())
    live_index = daily_returns.index[daily_returns.index >= pd.Timestamp(START)]
    rows = []
    active = pd.Series(dtype=float)
    last = None
    live = False
    for dt in live_index:
        prior = [d for d in rebal if pd.Timestamp(d) <= dt]
        if prior and prior[-1] != last:
            active = weights_by_date.get(prior[-1], pd.Series(dtype=float))
            last = prior[-1]
            if not active.empty:
                live = True
        if not live or active.empty:
            rows.append(np.nan)
            continue
        held = [s for s in active.index if s in daily_returns.columns]
        w = active.reindex(held).astype(float)
        w = w[w > 0]
        if w.empty:
            rows.append(0.0)
            continue
        w = w / w.sum()
        day_ret = (daily_returns.loc[dt, w.index] * w).sum(skipna=True)
        rows.append(0.0 if pd.isna(day_ret) else float(day_ret))
    return pd.Series(rows, index=live_index, dtype=float)


price_panel = _price_panel
daily_rets = price_panel.pct_change(fill_method=None)
screener = AAOIFIScreener(debt_threshold=DEBT_THRESHOLD)

# Monthly books: Halal names in each SUE quintile of *latest live print* (not only SUE>2)
quintile_weights: dict[int, dict[date, pd.Series]] = {q: {} for q in range(1, 6)}
rebalance_dates_q = sorted(pd.to_datetime(metrics["as_of"].dropna().unique()).date)
halal_sue = sue_events[sue_events["symbol"].isin(eligible_universe)].copy()

for as_of in rebalance_dates_q:
    snap = metrics[pd.to_datetime(metrics["as_of"]).dt.date == as_of].copy()
    if snap.empty:
        continue
    screened = screener.evaluate_compliance(snap)
    snap = snap.drop(columns=[c for c in ("debt_ratio", "cash_ratio", "receivables_ratio") if c in snap.columns])
    passed = snap.merge(screened, on="symbol", how="inner")
    passed = passed[passed["is_compliant"].fillna(False)].copy()
    live = latest_events_as_of(halal_sue, as_of)
    live = live.merge(passed[["symbol", "market_cap"]], on="symbol", how="inner")
    live = live[live["sue"].notna() & np.isfinite(live["sue"])]
    live["days_since_print"] = live["announce_date"].map(
        lambda d: trading_days_between(price_panel.index, d, as_of)
    )
    live = live[(live["days_since_print"] >= 1) & (live["days_since_print"] <= HOLD_TRADING_DAYS)]
    if len(live) < 10:
        continue
    live = live.copy()
    live["q"] = pd.qcut(live["sue"], 5, labels=False, duplicates="drop") + 1
    for q, g in live.groupby("q"):
        gw = assign_cap_weights(g)
        quintile_weights[int(q)][as_of] = gw.set_index("symbol")["weight"].astype(float)

q_panel = {}
for q, wmap in quintile_weights.items():
    if not wmap:
        continue
    q_panel[f"Q{q}"] = cap_weighted_returns(wmap, daily_rets)

if q_panel:
    q_df = pd.DataFrame(q_panel).dropna(how="all")
    q_equity = (1 + q_df).cumprod()
    q_equity.plot(figsize=(11, 4.5), title="SUE quintiles inside Halal names still in the 60-day window (cap-weighted)")
    plt.ylabel("Growth of $1")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(fig_dir / "quintile-growth.png", dpi=140)
    plt.show()

    q_stats = pd.DataFrame([calc_performance_stats(q_df[c], c) for c in q_df.columns]).set_index("Strategy")
    display(q_stats.style.format(formatters))

    if {"Q1", "Q5"}.issubset(q_df.columns):
        spread = (1 + q_df["Q5"]).cumprod() / (1 + q_df["Q1"]).cumprod() - 1
        spread.plot(figsize=(11, 3.5), color="tab:purple", title="Q5 − Q1 cumulative relative wealth (SUE quintiles)")
        plt.axhline(0, color="black", linewidth=0.8)
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(fig_dir / "quintile-spread.png", dpi=140)
        plt.show()
else:
    print("Not enough names to form SUE quintiles.")

# Sector mix of the live Halal PEAD book
if not selections.empty and not gics.empty:
    mix = selections.merge(gics.reset_index(), on="symbol", how="left")
    mix["gics_sector"] = mix["gics_sector"].fillna("Unclassified")
    sector_w = (
        mix.groupby(["as_of", "gics_sector"], as_index=False)["weight"].sum()
        .pivot(index="as_of", columns="gics_sector", values="weight")
        .fillna(0.0)
        .sort_index()
    )
    top_sectors = sector_w.mean().sort_values(ascending=False).head(8).index
    sector_w[top_sectors].plot.area(figsize=(11, 4), title="Halal PEAD book — GICS sector weights", alpha=0.85)
    plt.ylabel("Weight")
    plt.ylim(0, 1)
    plt.legend(loc="upper left", fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(fig_dir / "sector-mix.png", dpi=140)
    plt.show()
    print("Average sector weights:")
    print(sector_w.mean().sort_values(ascending=False).head(10).map(lambda x: f"{x:.1%}").to_string())


## Step 10 — Friction: turnover and trading costs

**What the code cell does:**
- Measures one-way turnover at each month-end rebalance (PEAD books can turn over fast as prints age out of the 60-day window).
- Applies `COST_BPS` round-trip costs to estimate net-of-friction performance.


In [ ]:
def one_way_turnover(old: dict[str, float], new: dict[str, float]) -> float:
    if not new:
        return 0.0
    if not old:
        return 1.0
    names = set(old) | set(new)
    return 0.5 * sum(abs(new.get(n, 0.0) - old.get(n, 0.0)) for n in names)


weight_history: list[tuple[date, dict[str, float]]] = []
for as_of, g in selections.groupby("as_of"):
    weight_history.append((as_of, g.set_index("symbol")["weight"].astype(float).to_dict()))
weight_history.sort(key=lambda x: x[0])

turnovers = []
prev = {}
for as_of, w in weight_history:
    t = one_way_turnover(prev, w)
    turnovers.append({"as_of": as_of, "turnover": t})
    prev = w

turn_df = pd.DataFrame(turnovers)
if not turn_df.empty:
    turn_df = turn_df.set_index("as_of")
    print(
        f"Average one-way turnover: {turn_df['turnover'].mean():.1%}  |  "
        f"median {turn_df['turnover'].median():.1%}"
    )
    turn_df["turnover"].plot(figsize=(11, 3), title="Month-end one-way turnover (Halal PEAD)", color="tab:orange")
    plt.ylabel("Turnover")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(fig_dir / "turnover.png", dpi=140)
    plt.show()

    cost_drag = turn_df["turnover"] * (COST_BPS / 10_000.0)
    net = strategy_returns.set_index("date")["return"].copy()
    net.index = pd.to_datetime(net.index)
    for as_of, drag in cost_drag.items():
        hits = net.index[net.index >= pd.Timestamp(as_of)]
        if len(hits):
            net.loc[hits[0]] = net.loc[hits[0]] - float(drag)
    net_panel = align_to_live(net.rename(STRATEGY_LABEL), {"SPY": bench_series["SPY"]})
    gross_stats = calc_performance_stats(main_panel[STRATEGY_LABEL], f"{STRATEGY_LABEL} (gross)")
    if STRATEGY_LABEL in net_panel.columns:
        net_stats = calc_performance_stats(net_panel[STRATEGY_LABEL], f"{STRATEGY_LABEL} (net {COST_BPS}bps)")
    else:
        net_stats = calc_performance_stats(net_panel["strategy"], f"{STRATEGY_LABEL} (net {COST_BPS}bps)")
    display(pd.DataFrame([gross_stats, net_stats]).set_index("Strategy").style.format(formatters))
else:
    print("No selection history for turnover.")


### Notes on methodology

- **SUE** is \((EPS_{\text{actual}} - EPS_{\text{consensus}}) / \sigma\), with \(\sigma\) equal to the sample standard deviation of that stock's **prior** quarterly forecast errors. A name enters only when SUE `> 2`. Yahoo earnings dates supply actual and estimated EPS; prints missing either side are dropped.
- **PEAD window** is 60 trading days after the announcement. The announcement session itself is not in the event-study CAR so the overnight jump does not define the drift.
- **Halal book:** sector screen plus AAOIFI debt, cash, and receivables tests at each month-end. **Broad book:** the same SUE rule on S&P 500 constituents with no financial-ratio overlay (the white-paper contrast).
- **Weights** are market-cap proportional (long-only). Months with no qualifying prints earn 0% that month once the strategy is live.
- **Benchmarks:** SPY (tradable S&P 500 ETF), `^GSPC` (S&P 500 index), SPUS (Halal large-cap ETF).
- Survivorship: the S&P 500 list is current; historical membership is not reconstructed. Analyst coverage on Yahoo is incomplete. Treat results as exploratory research, not a live mandate.


## Summary

| Component | Rule |
| --- | --- |
| Universe | S&P 500 → Halal sector screen (Halal book) vs full index list (broad book) |
| Debt screen | AAOIFI (`debt < 30%`, cash & receivables caps) — Halal book only |
| Signal | SUE \(= (EPS - \text{consensus}) / \sigma_{\text{prior UE}}\) |
| Entry | SUE `> 2` and print still inside 60 trading days |
| Portfolio | Cap-weighted, monthly rebalance, long-only |
| Window | 2020-01-01 → 2025-12-31 |
| Benchmarks | SPY, S&P 500 (`^GSPC`), SPUS |
| Focus metric | PEAD CAR (1 / 5 / 21 / 60d) Halal vs broad, plus live-book CAGR |

Re-run with `FAST_MODE = False` for the paper figures. The animated cell above is the primary visual demonstration of returns and volatility versus the three market benchmarks.
